In [1]:
!pip install pandas scikit-learn nltk

  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     --------------------------------- ------ 51.2/61.0 kB 1.3 MB/s eta 0:00:01
     ---------------------------------------- 61.0/61.0 kB 1.1 MB/s eta 0:00:00
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB ? eta 0:00:00
     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     ---------------------------------------- 57.7/57.7 kB 3.0 MB/s eta 0:00:00
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (1


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
file_path = "All_emotions.csv"

df = pd.read_csv(file_path)

print("shape:", df.shape)
df.head()

shape: (7261, 3)


,speaker,utterance,emotion
0,patient,I can feel the cold water on my teeth.,neutral
1,dentist,It's important to keep the area clean.,neutral
2,patient,I try to brush twice a day like you said.,neutral
3,dentist,You might feel a bit of pressure.,neutral
4,patient,Should I open wider?,neutral


In [5]:
print(df.columns.tolist())

['speaker', 'utterance', 'emotion']


In [6]:
print(df["speaker"].unique())
print(df["speaker"].value_counts())

['patient' 'dentist']
speaker
patient    3636
dentist    3625
Name: count, dtype: int64


In [7]:
df["speaker"] = df["speaker"].astype(str).str.lower().str.strip()

df = df[df["speaker"] == "patient"]

print("After filtering only patient data:", df.shape)
df.head()

After filtering only patient data: (3636, 3)


,speaker,utterance,emotion
0,patient,I can feel the cold water on my teeth.,neutral
2,patient,I try to brush twice a day like you said.,neutral
4,patient,Should I open wider?,neutral
6,patient,I am not paying for a mistake you made.,angry
8,patient,I can feel the cold water on my teeth.,neutral


In [8]:
df = df[["utterance", "emotion"]].copy()
df.head()

,utterance,emotion
0,I can feel the cold water on my teeth.,neutral
2,I try to brush twice a day like you said.,neutral
4,Should I open wider?,neutral
6,I am not paying for a mistake you made.,angry
8,I can feel the cold water on my teeth.,neutral


In [10]:
df.dropna(subset=["utterance", "emotion"], inplace=True)

df["utterance"] = df["utterance"].astype(str)
df["emotion"] = df["emotion"].astype(str)

df = df[df["utterance"].str.strip() != ""]
df = df[df["emotion"].str.strip() != ""]

print("Shape after removing missing and empty values:", df.shape)

Shape after removing missing and empty values: (3636, 2)


In [11]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["utterance"].apply(clean_text)

df = df[df["clean_text"].str.strip() != ""]

df[["utterance", "clean_text", "emotion"]].head()

,utterance,clean_text,emotion
0,I can feel the cold water on my teeth.,i can feel the cold water on my teeth,neutral
2,I try to brush twice a day like you said.,i try to brush twice a day like you said,neutral
4,Should I open wider?,should i open wider,neutral
6,I am not paying for a mistake you made.,i am not paying for a mistake you made,angry
8,I can feel the cold water on my teeth.,i can feel the cold water on my teeth,neutral


In [12]:
def tokenize(text):
    return text.split()

df["tokens"] = df["clean_text"].apply(tokenize)

df[["clean_text", "tokens"]].head()

,clean_text,tokens
0,i can feel the cold water on my teeth,"[i, can, feel, the, cold, water, on, my, teeth]"
2,i try to brush twice a day like you said,"[i, try, to, brush, twice, a, day, like, you, ..."
4,should i open wider,"[should, i, open, wider]"
6,i am not paying for a mistake you made,"[i, am, not, paying, for, a, mistake, you, made]"
8,i can feel the cold water on my teeth,"[i, can, feel, the, cold, water, on, my, teeth]"


In [13]:
X_text = df["clean_text"]
y = df["emotion"]

print("Number of samples:", len(X_text))
print("\nLabel distribution:")
print(y.value_counts())

Number of samples: 3636

Label distribution:
emotion
neutral    2856
disgust     247
fear        195
angry       156
sad          98
happy        84
Name: count, dtype: int64


In [14]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=1
)

X_tfidf = vectorizer.fit_transform(X_text)

print("TF-IDF shape:", X_tfidf.shape)
print("\nตัวอย่าง feature 20 ตัวแรก:")
print(vectorizer.get_feature_names_out()[:20])

TF-IDF shape: (3636, 3000)

ตัวอย่าง feature 20 ตัวแรก:
['able' 'able to' 'about' 'about how' 'about it' 'about looking'
 'about losing' 'about my' 'about needing' 'about that' 'about the'
 'about this' 'about to' 'about two' 'about what' 'about year'
 'absolutely' 'absolutely revolting' 'accident' 'ache']


In [15]:
tfidf_sample = pd.DataFrame(
    X_tfidf[:5].toarray(),
    columns=vectorizer.get_feature_names_out()
)

tfidf_sample.head()

,able,able to,about,about how,about it,about looking,about losing,about my,about needing,about that,...,you were,you will,you would,youll,youll see,your,your receptionist,your time,youre,youre right
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
df.to_csv("cleaned_emotions_patient.csv", index=False, encoding="utf-8-sig")

tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

tfidf_df["emotion"] = y.values

tfidf_df.to_csv("tfidf_features_patient.csv", index=False, encoding="utf-8-sig")

print("Files saved successfully:")
print("- cleaned_emotions_patient.csv")
print("- tfidf_features_patient.csv")

Files saved successfully:
- cleaned_emotions_patient.csv
- tfidf_features_patient.csv
